# Notebook 01 — Get the Canonical Course Dataset from GitHub
## Teaching PCD Environmental GIS

**สำหรับนิสิตปริญญาตรีปี 3–4 และปริญญาโท**

Repository:

```text
https://github.com/nattaponm/Teaching_PCD_Environmental_GIS
```

---

# บทบาทของ Notebook 01

Notebook 00 เป็น **Instructor Dataset Builder**

แต่ Notebook 01 เป็นจุดเริ่มต้นของนิสิต

```text
Teaching_PCD_Environmental_GIS
          ↓
dataset_packages.csv
          ↓
download canonical ZIP packages
          ↓
SHA256 verification
          ↓
safe extraction
          ↓
validate files / CRS / layers
          ↓
TEACHING DATASET READY
          ↓
Notebook 02–06
```

## หลักการสำคัญ

จาก Notebook นี้เป็นต้นไป เราจะใช้ **canonical teaching dataset**
ที่ผู้สอนได้รวบรวมและตรวจสอบไว้แล้ว

นักศึกษาไม่ต้องดาวน์โหลดข้อมูลจาก:

- Air4Thai โดยตรง
- OGIMET โดยตรง
- `prasertcbs/thailand_gis` โดยตรง

เหตุผลคือทุกคนควรใช้ **dataset version เดียวกัน**
เพื่อให้ผลการเรียน การทดลอง และการเปรียบเทียบทำซ้ำได้

# วัตถุประสงค์การเรียนรู้

เมื่อจบ Notebook นี้ นิสิตควรสามารถ:

1. อธิบายความหมายของ canonical dataset ได้
2. อธิบายหน้าที่ของ data manifest ได้
3. เข้าใจว่า SHA256 ใช้ตรวจ file integrity อย่างไร
4. ดาวน์โหลดข้อมูลจาก GitHub ด้วย Python ได้
5. extract ZIP อย่างปลอดภัยได้
6. ตรวจว่า raw Excel 2021–2025 มีครบหรือไม่
7. ตรวจ station metadata CSV ได้
8. เปิด GeoPackage ด้วย GeoPandas ได้
9. ตรวจ layers และ CRS ได้
10. อ่าน metadata, QC และ provenance report ได้
11. เข้าใจว่า “download สำเร็จ” ไม่เท่ากับ “dataset พร้อมวิเคราะห์”
12. เตรียม Google Drive ให้พร้อมสำหรับ Notebook 02–06

# 1.1 ข้อมูลที่คาดว่าจะได้รับ

Canonical dataset แบ่งเป็นสาม package

```text
Core
├── processed/
└── metadata/

PCD Excel
└── PCD_Excel/
    ├── 2021.xlsx
    ├── 2022.xlsx
    ├── 2023.xlsx
    ├── 2024.xlsx
    └── 2025.xlsx

GIS Source
└── GIS_source/
    ├── province source ZIP
    ├── amphoe source ZIP
    └── tambon source ZIP
```

ไฟล์สำคัญที่เราจะใช้ในบทต่อไป เช่น:

```text
pcd_air4thai_stations.csv
ogimet_wmo_stations_from_course_sources.csv
thailand_meteorological_stations_from_course_sources.csv
environmental_gis_course.gpkg
thailand_admin_lookup.csv
```

GeoPackage หลักควรมี layers:

```text
province
amphoe
tambon
pcd_stations
met_stations_thailand
ogimet_stations
```

In [1]:
# CELL 1 — Install libraries required by the course

!pip -q install -U \
    certifi \
    openpyxl \
    geopandas \
    pyogrio \
    shapely \
    pyproj \
    mapclassify \
    libpysal \
    esda \
    splot \
    folium

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.0/479.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.8/247.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 47.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.5 which is incompatible.


In [2]:
# CELL 2 — Mount Google Drive

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

Mounted at /content/drive


In [3]:
# CELL 3 — Imports

from pathlib import Path
import hashlib
import json
import shutil
import zipfile

import numpy as np
import pandas as pd
import requests
import geopandas as gpd

print("pandas   :", pd.__version__)
print("geopandas:", gpd.__version__)

pandas   : 3.0.5
geopandas: 1.1.4


# 1.2 Course Configuration

ใน course นี้จะใช้ dataset version `v1`

หากในอนาคตมีการปรับข้อมูล ผู้สอนสามารถสร้าง:

```text
v2
v3
...
```

โดยไม่เขียนทับ dataset เก่า

ดังนั้น version ของข้อมูลจึงเป็นส่วนหนึ่งของ reproducibility

In [4]:
# CELL 4 — Course configuration

DATASET_VERSION = "v1"

GITHUB_OWNER = "nattaponm"
GITHUB_REPO = "Teaching_PCD_Environmental_GIS"
GITHUB_BRANCH = "main"

GITHUB_RAW_BASE = (
    f"https://raw.githubusercontent.com/"
    f"{GITHUB_OWNER}/{GITHUB_REPO}/"
    f"{GITHUB_BRANCH}"
)

COURSE_REPOSITORY = (
    f"https://github.com/"
    f"{GITHUB_OWNER}/{GITHUB_REPO}"
)

print("Repository:")
print(COURSE_REPOSITORY)

print("\nDataset version:")
print(DATASET_VERSION)

Repository:
https://github.com/nattaponm/Teaching_PCD_Environmental_GIS

Dataset version:
v1


# 1.3 Google Drive Structure

Notebook นี้จะสร้าง:

```text
Teaching_PCD_Environmental_GIS/
│
├── 01_course_data/
│   └── v1/
│       ├── downloads/
│       └── dataset/
│
├── 02_output/
├── 03_figures/
└── 04_student_work/
```

หลักการ:

- `downloads/` = ZIP ที่ดาวน์โหลดจาก GitHub
- `dataset/` = ไฟล์ที่ extract แล้วและใช้ในการวิเคราะห์
- `02_output/` = ตารางผลวิเคราะห์
- `03_figures/` = รูป
- `04_student_work/` = งานทดลอง/แบบฝึกหัด

Notebook ต่อไปจะอ่านข้อมูลจาก `dataset/`

In [5]:
# CELL 5 — Create course folders

BASE_DIR = Path(
    "/content/drive/MyDrive/"
    "Teaching_PCD_Environmental_GIS"
)

VERSION_DIR = (
    BASE_DIR
    / "01_course_data"
    / DATASET_VERSION
)

DOWNLOAD_DIR = (
    VERSION_DIR
    / "downloads"
)

DATASET_DIR = (
    VERSION_DIR
    / "dataset"
)

OUTPUT_DIR = (
    BASE_DIR
    / "02_output"
)

FIGURE_DIR = (
    BASE_DIR
    / "03_figures"
)

STUDENT_WORK_DIR = (
    BASE_DIR
    / "04_student_work"
)

for folder in [
    DOWNLOAD_DIR,
    DATASET_DIR,
    OUTPUT_DIR,
    FIGURE_DIR,
    STUDENT_WORK_DIR,
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("Course root:")
print(BASE_DIR)

print("\nCanonical dataset folder:")
print(DATASET_DIR)

Course root:
/content/drive/MyDrive/Teaching_PCD_Environmental_GIS

Canonical dataset folder:
/content/drive/MyDrive/Teaching_PCD_Environmental_GIS/01_course_data/v1/dataset


# 1.4 Download the Package Manifest First

แทนที่จะเขียนชื่อ ZIP ทุกไฟล์ไว้ใน code เราจะให้ Notebook อ่าน:

```text
dataset_packages.csv
```

จาก repository

Manifest ควรบอกอย่างน้อย:

```text
package_type
filename
size_MB
sha256
required_for_students
```

นี่เป็นแนวคิดเดียวกับ **data catalog**

> Code ไม่ต้องเดาว่ามีไฟล์อะไร — ให้ manifest เป็นผู้บอก

In [6]:
# CELL 6 — Download manifest files from the course repository

MANIFEST_FILES = [
    "dataset_packages.csv",
    "dataset_packages.sha256",
    "final_build_summary.csv",
    "README_DATASET.md",
]

downloaded_manifest_files = []

for filename in MANIFEST_FILES:

    url = (
        GITHUB_RAW_BASE
        + "/"
        + filename
    )

    destination = (
        VERSION_DIR
        / filename
    )

    response = requests.get(
        url,
        timeout=60,
        headers={
            "User-Agent":
                "Teaching_PCD_Environmental_GIS"
        },
    )

    response.raise_for_status()

    destination.write_bytes(
        response.content
    )

    downloaded_manifest_files.append(
        destination
    )

    print(
        "Downloaded:",
        filename
    )

Downloaded: dataset_packages.csv
Downloaded: dataset_packages.sha256
Downloaded: final_build_summary.csv
Downloaded: README_DATASET.md


In [7]:
# CELL 7 — Read dataset_packages.csv

PACKAGE_MANIFEST = (
    VERSION_DIR
    / "dataset_packages.csv"
)

packages = pd.read_csv(
    PACKAGE_MANIFEST
)

print(
    "Package-manifest columns:"
)

print(
    list(
        packages.columns
    )
)

display(
    packages
)

Package-manifest columns:
['package_type', 'filename', 'size_MB', 'sha256', 'github_file_size_ok', 'required_for_students']


,package_type,filename,size_MB,sha256,github_file_size_ok,required_for_students
0,core,Teaching_PCD_Environmental_GIS_Core_v1.zip,6.841057,c9e335506b7c1a30335ead5e04a04d6c1ed871251ccb3f...,True,True
1,pcd_excel,Teaching_PCD_Environmental_GIS_PCD_Excel_v1.zip,0.999607,1639949d55c2f35e0bf8cd0666bad4e44c65144d03d769...,True,True
2,gis_source,Teaching_PCD_Environmental_GIS_GIS_Source_v1.zip,5.560163,41c23cff60de3d448085cfb71b28876b1ff43c86743840...,True,True


# 1.5 Read the Instructor Build Status

ก่อนดาวน์โหลดข้อมูล เราจะตรวจ:

```text
final_build_summary.csv
```

ถ้า instructor build มีสถานะ:

```text
GITHUB_READY
```

หมายความว่า automated build ไม่พบ `FAIL`
และ package size ผ่านข้อกำหนด

อย่างไรก็ตาม นักศึกษายังต้องตรวจข้อมูลหลังดาวน์โหลดอีกครั้ง

In [8]:
# CELL 8 — Inspect instructor build status

BUILD_SUMMARY_FILE = (
    VERSION_DIR
    / "final_build_summary.csv"
)

build_summary = pd.read_csv(
    BUILD_SUMMARY_FILE
)

display(
    build_summary
)

if (
    "status"
    in build_summary.columns
):

    build_status = (
        str(
            build_summary[
                "status"
            ].iloc[
                0
            ]
        )
        .strip()
    )

else:

    build_status = "UNKNOWN"


print(
    "\nInstructor build status:",
    build_status
)


if build_status != "GITHUB_READY":

    print(
        "\nWARNING:"
    )

    print(
        "The repository build summary is not "
        "marked GITHUB_READY."
    )

    print(
        "Continue only for inspection/teaching, "
        "and report this to the instructor."
    )

,dataset_version,build_utc,acceptance_fail_n,acceptance_check_n,oversize_package_n,status,github_ready_directory
0,v1,2026-08-22T11:31:27.298476+00:00,0,0,0,GITHUB_READY,/content/drive/MyDrive/Teaching_PCD_Environmen...



Instructor build status: GITHUB_READY


# 1.6 SHA256

SHA256 เป็น cryptographic hash ของไฟล์

แนวคิด:

```text
downloaded file
      ↓
calculate SHA256
      ↓
compare with manifest
      ↓
same?
```

ถ้า hash ไม่ตรง:

```text
file incomplete
หรือ
file changed
หรือ
wrong file/version
```

เราจะ **ไม่ unzip package ที่ SHA256 ไม่ตรง**

In [9]:
# CELL 9 — SHA256 helper

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    path = Path(
        path
    )

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        while True:

            chunk = f.read(
                chunk_size
            )

            if not chunk:
                break

            h.update(
                chunk
            )

    return h.hexdigest()


def file_size_mb(
    path
):
    return (
        Path(
            path
        ).stat().st_size
        / (1024 ** 2)
    )

# 1.7 Download Canonical Packages

Notebook จะเลือก rows ที่:

```text
required_for_students = True
```

ถ้ามีไฟล์อยู่แล้วและ SHA256 ถูกต้อง จะไม่ดาวน์โหลดซ้ำ

ถ้า hash ไม่ตรง จะดาวน์โหลดใหม่

In [10]:
# CELL 10 — Normalize manifest boolean field

if (
    "required_for_students"
    not in packages.columns
):

    raise KeyError(
        "dataset_packages.csv is missing "
        "'required_for_students'."
    )


required_text = (
    packages[
        "required_for_students"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
)

packages[
    "required_for_students_bool"
] = required_text.isin(
    [
        "true",
        "1",
        "yes",
        "y",
    ]
)


required_packages = (
    packages[
        packages[
            "required_for_students_bool"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


print(
    "Required packages:",
    len(
        required_packages
    )
)

display(
    required_packages
)

Required packages: 3


,package_type,filename,size_MB,sha256,github_file_size_ok,required_for_students,required_for_students_bool
0,core,Teaching_PCD_Environmental_GIS_Core_v1.zip,6.841057,c9e335506b7c1a30335ead5e04a04d6c1ed871251ccb3f...,True,True,True
1,pcd_excel,Teaching_PCD_Environmental_GIS_PCD_Excel_v1.zip,0.999607,1639949d55c2f35e0bf8cd0666bad4e44c65144d03d769...,True,True,True
2,gis_source,Teaching_PCD_Environmental_GIS_GIS_Source_v1.zip,5.560163,41c23cff60de3d448085cfb71b28876b1ff43c86743840...,True,True,True


In [11]:
# CELL 11 — Download and verify each canonical package

download_records = []

for _, row in required_packages.iterrows():

    filename = str(
        row[
            "filename"
        ]
    ).strip()

    expected_sha = str(
        row[
            "sha256"
        ]
    ).strip().lower()

    url = (
        GITHUB_RAW_BASE
        + "/"
        + filename
    )

    destination = (
        DOWNLOAD_DIR
        / filename
    )

    use_cached = False

    if (
        destination.exists()
        and destination.stat().st_size > 0
    ):

        cached_sha = sha256_file(
            destination
        )

        if (
            cached_sha.lower()
            == expected_sha
        ):

            use_cached = True

            print(
                "Verified cached:",
                filename
            )

        else:

            print(
                "Cached SHA256 mismatch — "
                "re-downloading:",
                filename
            )

            destination.unlink()


    if not use_cached:

        response = requests.get(
            url,
            timeout=300,
            headers={
                "User-Agent":
                    "Teaching_PCD_Environmental_GIS"
            },
        )

        response.raise_for_status()

        destination.write_bytes(
            response.content
        )


    actual_sha = sha256_file(
        destination
    )

    sha_ok = (
        actual_sha.lower()
        == expected_sha
    )

    download_records.append({
        "filename":
            filename,

        "size_MB":
            file_size_mb(
                destination
            ),

        "expected_sha256":
            expected_sha,

        "actual_sha256":
            actual_sha,

        "sha256_ok":
            sha_ok,

        "downloaded_path":
            str(
                destination
            ),
    })

    print(
        filename,
        "SHA256:",
        "PASS"
        if sha_ok
        else "FAIL"
    )


download_validation = pd.DataFrame(
    download_records
)

display(
    download_validation
)


if not download_validation[
    "sha256_ok"
].all():

    bad = download_validation[
        ~download_validation[
            "sha256_ok"
        ]
    ]

    raise RuntimeError(
        "SHA256 validation failed for: "
        + ", ".join(
            bad[
                "filename"
            ].tolist()
        )
    )

Teaching_PCD_Environmental_GIS_Core_v1.zip SHA256: PASS
Teaching_PCD_Environmental_GIS_PCD_Excel_v1.zip SHA256: PASS
Teaching_PCD_Environmental_GIS_GIS_Source_v1.zip SHA256: PASS


,filename,size_MB,expected_sha256,actual_sha256,sha256_ok,downloaded_path
0,Teaching_PCD_Environmental_GIS_Core_v1.zip,6.841057,c9e335506b7c1a30335ead5e04a04d6c1ed871251ccb3f...,c9e335506b7c1a30335ead5e04a04d6c1ed871251ccb3f...,True,/content/drive/MyDrive/Teaching_PCD_Environmen...
1,Teaching_PCD_Environmental_GIS_PCD_Excel_v1.zip,0.999607,1639949d55c2f35e0bf8cd0666bad4e44c65144d03d769...,1639949d55c2f35e0bf8cd0666bad4e44c65144d03d769...,True,/content/drive/MyDrive/Teaching_PCD_Environmen...
2,Teaching_PCD_Environmental_GIS_GIS_Source_v1.zip,5.560163,41c23cff60de3d448085cfb71b28876b1ff43c86743840...,41c23cff60de3d448085cfb71b28876b1ff43c86743840...,True,/content/drive/MyDrive/Teaching_PCD_Environmen...


# 1.8 Safe ZIP Extraction

ZIP สามารถมี path ที่ไม่ปลอดภัย เช่น:

```text
../../some_file
```

จึงไม่ควร `extractall()` โดยไม่ตรวจ

Notebook ใช้ safe extraction:

1. resolve destination path
2. ตรวจว่าอยู่ภายใน dataset folder
3. จึง extract

นี่เป็นตัวอย่างพื้นฐานของ **defensive data engineering**

In [12]:
# CELL 12 — Safe ZIP extraction helper

def safe_extract_zip(
    zip_path,
    destination,
):
    zip_path = Path(
        zip_path
    )

    destination = Path(
        destination
    )

    destination.mkdir(
        parents=True,
        exist_ok=True
    )

    destination_resolved = (
        destination.resolve()
    )

    with zipfile.ZipFile(
        zip_path,
        "r"
    ) as zf:

        for member in zf.infolist():

            target = (
                destination
                / member.filename
            ).resolve()

            try:

                target.relative_to(
                    destination_resolved
                )

            except ValueError:

                raise RuntimeError(
                    "Unsafe path found inside ZIP: "
                    + member.filename
                )

        zf.extractall(
            destination
        )

In [13]:
# CELL 13 — Extract all verified packages

# Recreate the extracted dataset to avoid mixing old/new versions.
if DATASET_DIR.exists():

    shutil.rmtree(
        DATASET_DIR
    )

DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)


extract_records = []

for _, row in download_validation.iterrows():

    zip_path = Path(
        row[
            "downloaded_path"
        ]
    )

    if zip_path.suffix.lower() != ".zip":

        continue

    safe_extract_zip(
        zip_path,
        DATASET_DIR
    )

    extract_records.append({
        "filename":
            zip_path.name,

        "status":
            "EXTRACTED",
    })

    print(
        "Extracted:",
        zip_path.name
    )


extract_summary = pd.DataFrame(
    extract_records
)

display(
    extract_summary
)

Extracted: Teaching_PCD_Environmental_GIS_Core_v1.zip
Extracted: Teaching_PCD_Environmental_GIS_PCD_Excel_v1.zip
Extracted: Teaching_PCD_Environmental_GIS_GIS_Source_v1.zip


,filename,status
0,Teaching_PCD_Environmental_GIS_Core_v1.zip,EXTRACTED
1,Teaching_PCD_Environmental_GIS_PCD_Excel_v1.zip,EXTRACTED
2,Teaching_PCD_Environmental_GIS_GIS_Source_v1.zip,EXTRACTED


# 1.9 Dataset File Inventory

หลัง unzip แล้ว ต้องดู **ของจริงใน folder**

ไม่ควรคิดว่า:

> unzip สำเร็จ = ทุกไฟล์ที่ต้องใช้มีครบ

เราจะสร้าง inventory:

```text
relative_path
suffix
size_MB
```

In [14]:
# CELL 14 — Build extracted-file inventory

inventory_rows = []

for path in DATASET_DIR.rglob(
    "*"
):

    if not path.is_file():
        continue

    inventory_rows.append({
        "filename":
            path.name,

        "relative_path":
            str(
                path.relative_to(
                    DATASET_DIR
                )
            ),

        "suffix":
            path.suffix.lower(),

        "size_MB":
            file_size_mb(
                path
            ),
    })


dataset_file_inventory = (
    pd.DataFrame(
        inventory_rows
    )
    .sort_values(
        "relative_path"
    )
    .reset_index(
        drop=True
    )
)

print(
    "Extracted files:",
    len(
        dataset_file_inventory
    )
)

display(
    dataset_file_inventory
)

Extracted files: 35


,filename,relative_path,suffix,size_MB
0,amphoe_simplify_source.zip,GIS_source/amphoe_simplify_source.zip,.zip,1.728478
1,province_simplify_source.zip,GIS_source/province_simplify_source.zip,.zip,1.705416
2,tambon_simplify_source.zip,GIS_source/tambon_simplify_source.zip,.zip,2.183973
3,2021.xlsx,PCD_Excel/2021.xlsx,.xlsx,0.165023
4,2022.xlsx,PCD_Excel/2022.xlsx,.xlsx,0.186507
5,2023.xlsx,PCD_Excel/2023.xlsx,.xlsx,0.229081
6,2024.xlsx,PCD_Excel/2024.xlsx,.xlsx,0.237037
7,2025.xlsx,PCD_Excel/2025.xlsx,.xlsx,0.221927
8,README_DATASET.md,README_DATASET.md,.md,0.001700
9,00_spatial_QC_map.png,metadata/00_spatial_QC_map.png,.png,0.440454


# 1.10 Locate Canonical Folders Automatically

ZIP package อาจเก็บ path เช่น:

```text
processed/...
metadata/...
PCD_Excel/...
GIS_source/...
```

เราจะค้น folder/file ด้วยชื่อแทนการสมมติ path ที่ลึกเกินไป

In [15]:
# CELL 15 — Locate important canonical files

def find_unique_file(
    root,
    filename,
    required=True,
):
    matches = list(
        Path(
            root
        ).rglob(
            filename
        )
    )

    if len(
        matches
    ) == 1:

        return matches[
            0
        ]

    if len(
        matches
    ) == 0:

        if required:

            raise FileNotFoundError(
                f"Required file not found: {filename}"
            )

        return None

    raise RuntimeError(
        f"Multiple copies found for {filename}: "
        + str(
            matches
        )
    )


PCD_STATION_CSV = find_unique_file(
    DATASET_DIR,
    "pcd_air4thai_stations.csv",
)

OGIMET_STATION_CSV = find_unique_file(
    DATASET_DIR,
    (
        "ogimet_wmo_stations_"
        "from_course_sources.csv"
    ),
)

THAILAND_MET_STATION_CSV = (
    find_unique_file(
        DATASET_DIR,
        (
            "thailand_meteorological_stations_"
            "from_course_sources.csv"
        ),
        required=False,
    )
)

COURSE_GPKG = find_unique_file(
    DATASET_DIR,
    "environmental_gis_course.gpkg",
)

ADMIN_LOOKUP_CSV = find_unique_file(
    DATASET_DIR,
    "thailand_admin_lookup.csv",
)

DATA_PROVENANCE_CSV = find_unique_file(
    DATASET_DIR,
    "data_provenance.csv",
)

ACCEPTANCE_TEST_CSV = find_unique_file(
    DATASET_DIR,
    "acceptance_test.csv",
)


print("PCD stations :", PCD_STATION_CSV)
print("OGIMET/WMO   :", OGIMET_STATION_CSV)
print("Met network  :", THAILAND_MET_STATION_CSV)
print("GeoPackage   :", COURSE_GPKG)
print("Admin lookup :", ADMIN_LOOKUP_CSV)

PCD stations : /content/drive/MyDrive/Teaching_PCD_Environmental_GIS/01_course_data/v1/dataset/processed/pcd_air4thai_stations.csv
OGIMET/WMO   : /content/drive/MyDrive/Teaching_PCD_Environmental_GIS/01_course_data/v1/dataset/processed/ogimet_wmo_stations_from_course_sources.csv
Met network  : /content/drive/MyDrive/Teaching_PCD_Environmental_GIS/01_course_data/v1/dataset/processed/thailand_meteorological_stations_from_course_sources.csv
GeoPackage   : /content/drive/MyDrive/Teaching_PCD_Environmental_GIS/01_course_data/v1/dataset/processed/environmental_gis_course.gpkg
Admin lookup : /content/drive/MyDrive/Teaching_PCD_Environmental_GIS/01_course_data/v1/dataset/processed/thailand_admin_lookup.csv


# 1.11 Validate PCD Excel 2021–2025

Raw Excel เป็นข้อมูลสำคัญสำหรับ Notebook 02

ตรวจ:

```text
2021.xlsx
2022.xlsx
2023.xlsx
2024.xlsx
2025.xlsx
```

และทดลองเปิด workbook เพื่อดู sheet names

In [16]:
# CELL 16 — Validate PCD Excel files

EXPECTED_YEARS = [
    2021,
    2022,
    2023,
    2024,
    2025,
]

excel_validation_rows = []

for year in EXPECTED_YEARS:

    filename = (
        f"{year}.xlsx"
    )

    matches = list(
        DATASET_DIR.rglob(
            filename
        )
    )

    if len(
        matches
    ) != 1:

        excel_validation_rows.append({
            "year":
                year,

            "status":
                "FAIL",

            "file_matches":
                len(
                    matches
                ),

            "path":
                None,
        })

        continue

    path = matches[
        0
    ]

    try:

        workbook = pd.ExcelFile(
            path
        )

        sheets = workbook.sheet_names

        excel_validation_rows.append({
            "year":
                year,

            "status":
                "PASS",

            "file_matches":
                1,

            "size_MB":
                file_size_mb(
                    path
                ),

            "sheet_n":
                len(
                    sheets
                ),

            "sheet_names":
                " | ".join(
                    sheets
                ),

            "path":
                str(
                    path
                ),
        })

    except Exception as exc:

        excel_validation_rows.append({
            "year":
                year,

            "status":
                "FAIL",

            "file_matches":
                1,

            "error":
                str(
                    exc
                ),

            "path":
                str(
                    path
                ),
        })


excel_validation = pd.DataFrame(
    excel_validation_rows
)

display(
    excel_validation
)

,year,status,file_matches,size_MB,sheet_n,sheet_names,path
0,2021,PASS,1,0.165023,2,PM2.5 | พารามิเตอร์_สถานี,/content/drive/MyDrive/Teaching_PCD_Environmen...
1,2022,PASS,1,0.186507,2,PM2.5 | พารามิเตอร์_สถานี,/content/drive/MyDrive/Teaching_PCD_Environmen...
2,2023,PASS,1,0.229081,2,PM2.5 | รายละเอียดจุดตรวจวัด,/content/drive/MyDrive/Teaching_PCD_Environmen...
3,2024,PASS,1,0.237037,2,Data | รายละเอียดจุดตรวจวัด,/content/drive/MyDrive/Teaching_PCD_Environmen...
4,2025,PASS,1,0.221927,2,DATA | รายละเอียดจุดตรวจวัด,/content/drive/MyDrive/Teaching_PCD_Environmen...


# 1.12 Validate Station Metadata CSV

PCD station table ควรมีอย่างน้อย:

```text
station_id
latitude
longitude
```

OGIMET/WMO table ควรมี:

```text
wmo_id
latitude
longitude
```

เราจะตรวจ:

- unique station ID
- missing coordinate
- numeric coordinate
- broad Thailand range

In [17]:
# CELL 17 — PCD station validation

pcd_stations = pd.read_csv(
    PCD_STATION_CSV
)

PCD_REQUIRED_FIELDS = [
    "station_id",
    "latitude",
    "longitude",
]

pcd_missing_fields = [
    field
    for field
    in PCD_REQUIRED_FIELDS
    if field
    not in pcd_stations.columns
]

if pcd_missing_fields:

    raise KeyError(
        "PCD station table is missing: "
        + ", ".join(
            pcd_missing_fields
        )
    )


pcd_stations[
    "latitude"
] = pd.to_numeric(
    pcd_stations[
        "latitude"
    ],
    errors="coerce",
)

pcd_stations[
    "longitude"
] = pd.to_numeric(
    pcd_stations[
        "longitude"
    ],
    errors="coerce",
)


pcd_coordinate_valid = (
    pcd_stations[
        "latitude"
    ].between(
        5,
        21,
    )
    &
    pcd_stations[
        "longitude"
    ].between(
        96,
        106,
    )
)


pcd_validation = pd.DataFrame([
    {
        "check":
            "rows",

        "value":
            len(
                pcd_stations
            ),
    },

    {
        "check":
            "unique_station_id",

        "value":
            pcd_stations[
                "station_id"
            ].nunique(),
    },

    {
        "check":
            "duplicate_station_id_rows",

        "value":
            int(
                pcd_stations[
                    "station_id"
                ]
                .duplicated(
                    keep=False
                )
                .sum()
            ),
    },

    {
        "check":
            "valid_coordinate_rows",

        "value":
            int(
                pcd_coordinate_valid.sum()
            ),
    },

    {
        "check":
            "invalid_coordinate_rows",

        "value":
            int(
                (
                    ~pcd_coordinate_valid
                ).sum()
            ),
    },
])


display(
    pcd_validation
)

display(
    pcd_stations.head(
        10
    )
)

,check,value
0,rows,173
1,unique_station_id,173
2,duplicate_station_id_rows,0
3,valid_coordinate_rows,173
4,invalid_coordinate_rows,0


,station_id,station_name_th,station_name_en,area_th,area_en,station_type,latitude,longitude,coordinate_source_type,coordinate_source_file,coordinate_valid,province_code,province_name_en,province_name_th,amphoe_code,amphoe_name_en,amphoe_name_th,tambon_code,tambon_name_en,tambon_name_th
0,119t,สวนสาธารณะธารา,Thara Public Park,"ต.ปากน้ำ อ.เมือง, กระบี่","Pak Nam Subdistrict, Mueang District, Krabi",GROUND,8.050624,98.918049,Air4Thai_public_station_metadata,https://air4thai.pcd.go.th/services/getNewAQI_...,True,NaN,Krabi,กระบี่,TH8101,Mueang Krabi,เมืองกระบี่,TH810115,Sai Thai,à¹à¸ªà¹à¸à¸¢
1,02t,มหาวิทยาลัยราชภัฏบ้านสมเด็จเจ้าพระยา,Bansomdejchaopraya Rajabhat University,"แขวงหิรัญรูจี เขตธนบุรี, กรุงเทพฯ","Hiran Ruchi, Khet Thon Buri, Bangkok",GROUND,13.732846,100.487662,Air4Thai_public_station_metadata,https://air4thai.pcd.go.th/services/getNewAQI_...,True,NaN,Bangkok,กรุงเทพมหานคร,TH1015,Thon Buri,ธนบุรี,TH101502,Hiranruchi,à¸«à¸´à¸£à¸±à¸à¸£à¸¹à¸à¸µ
2,03t,ริมถนนทางหลวงหมายเลข 3902,Highway NO.3902 km.13 +600,"ริมถนนกาญจนาภิเษก เขตบางขุนเทียน, กรุงเทพฯ","Kanchanaphisek Rd, Bang Khun Thian, Bangkok",GROUND,13.636514,100.414262,Air4Thai_public_station_metadata,https://air4thai.pcd.go.th/services/getNewAQI_...,True,NaN,Bangkok,กรุงเทพมหานคร,TH1021,Bang Khun Thian,บางขุนเทียน,TH102107,Samae Dam,à¹à¸ªà¸¡à¸à¸³
3,05t,กรมอุตุนิยมวิทยาบางนา,Thai Meteorological Department,"แขวงบางนา เขตบางนา, กรุงเทพฯ","Bang Na, Khet Bang Na, Bangkok",GROUND,13.666183,100.605742,Air4Thai_public_station_metadata,https://air4thai.pcd.go.th/services/getNewAQI_...,True,NaN,Bangkok,กรุงเทพมหานคร,TH1047,Bang Na,บางนา,TH104701,Bang Na,à¸à¸²à¸à¸à¸²
4,12t,โรงเรียนนนทรีวิทยา,Nonsi Witthaya School,"แขวงช่องนนทรี เขตยานนาวา, กรุงเทพฯ","Chong Nonsi, Khet Yannawa, Bangkok",GROUND,13.708067,100.547333,Air4Thai_public_station_metadata,https://air4thai.pcd.go.th/services/getNewAQI_...,True,NaN,Bangkok,กรุงเทพมหานคร,TH1012,Yan Nawa,ยานนาวา,TH101203,Chong Nonsi,à¸à¹à¸­à¸à¸à¸à¸à¸£à¸µ
5,50t,โรงพยาบาลจุฬาลงกรณ์,Chulalongkorn Hospital,"ริมถนนพระราม 4 เขตปทุมวัน, กรุงเทพฯ","Rama IV Rd. Khet Pathum Wan, Bangkok",GROUND,13.729852,100.536501,Air4Thai_public_station_metadata,https://air4thai.pcd.go.th/services/getNewAQI_...,True,NaN,Bangkok,กรุงเทพมหานคร,TH1007,Pathum Wan,ปทุมวัน,TH100703,Pathum Wan,à¸à¸à¸¸à¸¡à¸§à¸±à¸
6,52t,การไฟฟ้าย่อยธนบุรี,Thonburi Power Sub-Station,"ริมถนนอินทรพิทักษ์ เขตธนบุรี, กรุงเทพฯ","Intarapitak Rd. Khet Thon Buri, Bangkok",GROUND,13.727622,100.486568,Air4Thai_public_station_metadata,https://air4thai.pcd.go.th/services/getNewAQI_...,True,NaN,Bangkok,กรุงเทพมหานคร,TH1015,Thon Buri,ธนบุรี,TH101502,Hiranruchi,à¸«à¸´à¸£à¸±à¸à¸£à¸¹à¸à¸µ
7,53t,สถานีตำรวจนครบาลโชคชัย,Chokchai Police Station,"ริมถนนลาดพร้าว เขตวังทองหลาง, กรุงเทพฯ","Lat Phrao Rd. Khet Wang Thonglang, Bangkok",GROUND,13.795425,100.593030,Air4Thai_public_station_metadata,https://air4thai.pcd.go.th/services/getNewAQI_...,True,NaN,Bangkok,กรุงเทพมหานคร,TH1045,Wang Thonglang,วังทองหลาง,TH104502,Saphan Song,à¸ªà¸°à¸à¸²à¸à¸ªà¸­à¸
8,54t,การเคหะชุมชนดินแดง,National Housing Authority Dindaeng,"ริมถนนดินแดง เขตดินแดง, กรุงเทพฯ","Din Daeng Rd. Khet Din Daeng, Bangkok",GROUND,13.762517,100.550200,Air4Thai_public_station_metadata,https://air4thai.pcd.go.th/services/getNewAQI_...,True,NaN,Bangkok,กรุงเทพมหานคร,TH1026,Din Daeng,ดินแดง,TH102601,Din Daeng,à¸à¸´à¸à¹à¸à¸
9,59t,กรมประชาสัมพันธ์,The Government Public Relations Department,"แขวงพญาไท เขตพญาไท, กรุงเทพฯ","Phaya Thai, Khet Phaya Thai, Bangkok",GROUND,13.783185,100.540489,Air4Thai_public_station_metadata,https://air4thai.pcd.go.th/services/getNewAQI_...,True,NaN,Bangkok,กรุงเทพมหานคร,TH1014,Phaya Thai,พญาไท,TH101401,Sam Sen Nai,à¸ªà¸²à¸¡à¹à¸ªà¸à¹à¸


In [18]:
# CELL 18 — OGIMET/WMO station validation

ogimet_stations = pd.read_csv(
    OGIMET_STATION_CSV,
    dtype={
        "wmo_id":
            "string"
    },
)

OGIMET_REQUIRED_FIELDS = [
    "wmo_id",
    "latitude",
    "longitude",
]

ogimet_missing_fields = [
    field
    for field
    in OGIMET_REQUIRED_FIELDS
    if field
    not in ogimet_stations.columns
]


if ogimet_missing_fields:

    raise KeyError(
        "OGIMET station table is missing: "
        + ", ".join(
            ogimet_missing_fields
        )
    )


ogimet_stations[
    "latitude"
] = pd.to_numeric(
    ogimet_stations[
        "latitude"
    ],
    errors="coerce",
)

ogimet_stations[
    "longitude"
] = pd.to_numeric(
    ogimet_stations[
        "longitude"
    ],
    errors="coerce",
)


ogimet_coordinate_valid = (
    ogimet_stations[
        "latitude"
    ].between(
        5,
        21,
    )
    &
    ogimet_stations[
        "longitude"
    ].between(
        96,
        106,
    )
)


ogimet_validation = pd.DataFrame([
    {
        "check":
            "rows",

        "value":
            len(
                ogimet_stations
            ),
    },

    {
        "check":
            "unique_wmo_id",

        "value":
            ogimet_stations[
                "wmo_id"
            ].nunique(),
    },

    {
        "check":
            "duplicate_wmo_id_rows",

        "value":
            int(
                ogimet_stations[
                    "wmo_id"
                ]
                .duplicated(
                    keep=False
                )
                .sum()
            ),
    },

    {
        "check":
            "valid_coordinate_rows",

        "value":
            int(
                ogimet_coordinate_valid.sum()
            ),
    },

    {
        "check":
            "invalid_coordinate_rows",

        "value":
            int(
                (
                    ~ogimet_coordinate_valid
                ).sum()
            ),
    },
])


display(
    ogimet_validation
)

display(
    ogimet_stations.head(
        10
    )
)

,check,value
0,rows,130
1,unique_wmo_id,130
2,duplicate_wmo_id_rows,0
3,valid_coordinate_rows,130
4,invalid_coordinate_rows,0


,wigos_id,wmo_id,icao_id,station_name,country,latitude,longitude,elevation_m,established,closed,...,province_code,province_name_en,province_name_th,amphoe_code,amphoe_name_en,amphoe_name_th,tambon_code,tambon_name_en,tambon_name_th,network_resolution
0,0-20000-0-48300,48300,VTCH,Mae Hong Son,Thailand,19.298889,97.975556,265,2014-11-01,----,...,NaN,Mae Hong Son,แม่ฮ่องสอน,TH5801,Mueang Mae Hong Son,เมืองแม่ฮ่องสอน,TH580101,Chong Kham,à¸à¸­à¸à¸à¸³,thailand_met_station_active
1,0-20000-0-48302,48302,----,Doi Ang Khang,Thailand,19.932778,99.045278,1529,1960-01-01,----,...,NaN,Chiang Mai,เชียงใหม่,TH5009,Fang,ฝาง,TH500903,Mon Pin,à¸¡à¹à¸­à¸à¸à¸´à¹à¸,thailand_met_station_active
2,0-20000-0-48303,48303,VTCT,Chiang Rai,Thailand,19.961389,99.880833,390,1951-01-01,----,...,NaN,Chiang Rai,เชียงราย,TH5701,Mueang Chiang Rai,เมืองเชียงราย,TH570103,Ban Du,à¸à¹à¸²à¸à¸à¸¹à¹,thailand_met_station_active
3,0-20000-0-48304,48304,----,Chaing Rai Agromet,Thailand,19.872222,99.779167,397,1978-12-01,----,...,NaN,Chiang Rai,เชียงราย,TH5701,Mueang Chiang Rai,เมืองเชียงราย,TH570116,Pa O Don Chai,à¸à¹à¸²à¸­à¹à¸­à¸à¸­à¸à¸à¸±à¸¢,thailand_met_station_active
4,0-20000-0-48307,48307,----,Tung Chang,Thailand,19.408056,100.881944,333,2001-01-01,----,...,NaN,Nan,น่าน,TH5508,Thung Chang,ทุ่งช้าง,TH550803,Lae,à¹à¸¥à¸°,thailand_met_station_active
5,0-20000-0-48310,48310,----,Phayao,Thailand,19.193056,99.883333,397,1982-01-01,----,...,NaN,Phayao,พะเยา,TH5601,Mueang Phayao,เมืองพะเยา,TH560107,Ban Tom,à¸à¹à¸²à¸à¸à¹à¸­à¸¡,thailand_met_station_active
6,0-20000-0-48315,48315,----,Tha Wang Pha,Thailand,19.123056,100.813056,235,1987-01-01,----,...,NaN,Nan,น่าน,TH5506,Tha Wang Pha,ท่าวังผา,TH550609,Tha Wang Pha,à¸à¹à¸²à¸§à¸±à¸à¸à¸²,thailand_met_station_active
7,0-20000-0-48324,48324,----,Thoen,Thailand,17.636389,99.244444,191,2003-01-01,----,...,NaN,Lampang,ลำปาง,TH5208,Thoen,เถิน,TH520801,Lom Raet,à¸¥à¹à¸­à¸¡à¹à¸£à¸,thailand_met_station_active
8,0-20000-0-48325,48325,VTCS,Mae Sariang,Thailand,18.175000,97.933333,212,1952-01-01,----,...,NaN,Mae Hong Son,แม่ฮ่องสอน,TH5804,Mae Sariang,แม่สะเรียง,TH580401,Ban Kat,à¸à¹à¸²à¸à¸à¸²à¸¨,thailand_met_station_active
9,0-0-0-MISSING,48326,----,Mae Jo Agromet,Thailand,18.916667,99.000000,317,----,----,...,NaN,Chiang Mai,เชียงใหม่,TH5014,San Sai,สันทราย,TH501408,Nong Han,à¸«à¸à¸­à¸à¸«à¸²à¸£,thailand_met_station_active


# 1.13 GeoPackage

GeoPackage (`.gpkg`) เป็น spatial database แบบไฟล์เดียว

ต่างจาก shapefile ที่มักต้องมีหลายไฟล์:

```text
.shp
.shx
.dbf
.prj
...
```

Course นี้ใช้ GeoPackage เพื่อให้ง่ายต่อการเรียนและลดปัญหาไฟล์ component สูญหาย

เราจะตรวจ:

- layer names
- feature count
- geometry type
- CRS
- invalid geometry

In [19]:
# CELL 19 — Inspect GeoPackage layers

import pyogrio

gpkg_layers_info = pyogrio.list_layers(
    COURSE_GPKG
)

gpkg_layers = [
    str(
        row[
            0
        ]
    )
    for row
    in gpkg_layers_info
]

print(
    "GeoPackage layers:"
)

for layer in gpkg_layers:

    print(
        " -",
        layer
    )

GeoPackage layers:
 - province
 - amphoe
 - tambon
 - pcd_stations
 - met_stations_thailand
 - ogimet_stations


In [20]:
# CELL 20 — Validate expected GeoPackage layers

EXPECTED_LAYERS = [
    "province",
    "amphoe",
    "tambon",
    "pcd_stations",
    "met_stations_thailand",
    "ogimet_stations",
]


layer_validation_rows = []


for layer in EXPECTED_LAYERS:

    if layer not in gpkg_layers:

        layer_validation_rows.append({
            "layer":
                layer,

            "status":
                "MISSING",
        })

        continue

    gdf = gpd.read_file(
        COURSE_GPKG,
        layer=layer,
    )

    geometry_types = (
        gdf.geometry
        .geom_type
        .value_counts(
            dropna=False
        )
        .to_dict()
    )

    invalid_n = int(
        (
            ~gdf.geometry.is_valid
        )
        .fillna(
            True
        )
        .sum()
    )

    layer_validation_rows.append({
        "layer":
            layer,

        "status":
            "PASS",

        "feature_n":
            len(
                gdf
            ),

        "crs":
            str(
                gdf.crs
            ),

        "geometry_types":
            str(
                geometry_types
            ),

        "invalid_geometry_n":
            invalid_n,
    })


layer_validation = pd.DataFrame(
    layer_validation_rows
)

display(
    layer_validation
)

,layer,status,feature_n,crs,geometry_types,invalid_geometry_n
0,province,PASS,77,EPSG:4326,{'MultiPolygon': 77},0
1,amphoe,PASS,928,EPSG:4326,{'MultiPolygon': 928},0
2,tambon,PASS,7425,EPSG:4326,{'MultiPolygon': 7425},0
3,pcd_stations,PASS,173,EPSG:4326,{'Point': 173},0
4,met_stations_thailand,PASS,140,EPSG:4326,{'Point': 140},0
5,ogimet_stations,PASS,130,EPSG:4326,{'Point': 130},0


# 1.14 First Look at Point and Polygon Data

ยังไม่วิเคราะห์ GIS ในบทนี้

เพียงเปิดตัวอย่างเพื่อให้เห็นว่า:

```text
PCD station = Point
OGIMET station = Point
Province = Polygon
Amphoe = Polygon
Tambon = Polygon
```

In [21]:
# CELL 21 — Read sample layers

province = gpd.read_file(
    COURSE_GPKG,
    layer="province",
)

pcd_points = gpd.read_file(
    COURSE_GPKG,
    layer="pcd_stations",
)

ogimet_points = gpd.read_file(
    COURSE_GPKG,
    layer="ogimet_stations",
)


print(
    "Province geometry:"
)

print(
    province.geometry.geom_type.value_counts()
)

print(
    "\nPCD geometry:"
)

print(
    pcd_points.geometry.geom_type.value_counts()
)

print(
    "\nOGIMET geometry:"
)

print(
    ogimet_points.geometry.geom_type.value_counts()
)


print(
    "\nShared CRS:"
)

print(
    "Province:",
    province.crs
)

print(
    "PCD     :",
    pcd_points.crs
)

print(
    "OGIMET  :",
    ogimet_points.crs
)

Province geometry:
MultiPolygon    77
Name: count, dtype: int64

PCD geometry:
Point    173
Name: count, dtype: int64

OGIMET geometry:
Point    130
Name: count, dtype: int64

Shared CRS:
Province: EPSG:4326
PCD     : EPSG:4326
OGIMET  : EPSG:4326


# 1.15 Provenance

`data_provenance.csv` ตอบคำถามสำคัญว่า:

> ข้อมูลนี้มาจากไหน?

ตัวอย่าง:

```text
PCD Excel
→ nattaponm/training_PCD_data_GIS

PCD station metadata
→ existing course source / Air4Thai metadata during instructor build

Thailand GIS
→ prasertcbs/thailand_gis
```

Data provenance เป็นส่วนหนึ่งของงานวิจัย ไม่ใช่รายละเอียดเสริม

In [22]:
# CELL 22 — Read provenance

data_provenance = pd.read_csv(
    DATA_PROVENANCE_CSV
)

display(
    data_provenance
)

,dataset,primary_source,source_location,processing,canonical_output,build_utc
0,PCD Excel 2021–2025,nattaponm/training_PCD_data_GIS,GitHub files 2021.xlsx–2025.xlsx,Copied unchanged,raw_reference/PCD_Excel,2026-08-22T11:31:27.298476+00:00
1,PCD/Air4Thai station metadata,Existing course GitHub sources; Air4Thai publi...,https://air4thai.pcd.go.th/services/getNewAQI_...,coordinate QC; deduplication; spatial join to ...,processed/pcd_air4thai_stations.csv,2026-08-22T11:31:27.298476+00:00
2,OGIMET/WMO station metadata,nattaponm/training_PCD_Ogimet10yr_3provinces_pm25,station metadata discovered in course data/code,table/literal extraction; coordinate QC; dedup...,processed/ogimet_wmo_stations_from_course_sour...,2026-08-22T11:31:27.298476+00:00
3,Thailand province boundaries,prasertcbs/thailand_gis,province/province_simplify.zip,extract simplified shapefile; geometry validat...,processed/environmental_gis_course.gpkg:province,2026-08-22T11:31:27.298476+00:00
4,Thailand amphoe boundaries,prasertcbs/thailand_gis,amphoe/thailand_province_amphoe_simplify.zip,extract simplified shapefile; geometry validat...,processed/environmental_gis_course.gpkg:amphoe,2026-08-22T11:31:27.298476+00:00
5,Thailand tambon boundaries,prasertcbs/thailand_gis,tambon_simplify/tha_admbnda_adm3_rtsd_20220121...,extract simplified shapefile; geometry validat...,processed/environmental_gis_course.gpkg:tambon,2026-08-22T11:31:27.298476+00:00


# 1.16 Instructor Acceptance Report

ข้อมูล canonical ควรมี acceptance report ที่สร้างจาก Notebook 00

นิสิตควรรู้ว่า:

- QC ถูกทำอะไรไปแล้ว?
- automated test ผ่านอะไร?
- มีจุดใดเป็น `CHECK` หรือไม่?

แต่ต้องจำว่า:

> QC ของผู้สอนไม่ได้แทน QC ของการวิเคราะห์ในงานวิจัย

Notebook ต่อ ๆ ไปยังต้องตรวจ data quality ตามโจทย์วิเคราะห์อีกครั้ง

In [23]:
# CELL 23 — Read acceptance report

acceptance_test = pd.read_csv(
    ACCEPTANCE_TEST_CSV
)

display(
    acceptance_test
)

,test,status,detail
0,PCD Excel 2021–2025,PASS,5/5 files
1,PCD station IDs unique,PASS,173 rows
2,PCD stations matched to province,PASS,100.0%
3,OGIMET station metadata available,PASS,130 station(s)
4,OGIMET WMO IDs unique,PASS,130 unique IDs
5,Province feature count,PASS,77
6,GIS invalid geometries after repair,PASS,0
7,GeoPackage layers re-open,PASS,6/6
8,Canonical CRS = EPSG:4326,PASS,EPSG:4326


# 1.17 Student Dataset Summary

สรุปข้อมูลที่นิสิตมีอยู่ก่อนเริ่มวิเคราะห์

นี่เป็น **inventory**, ยังไม่ใช่ผลการวิจัย

In [24]:
# CELL 24 — Dataset summary

province_n = (
    int(
        layer_validation.loc[
            layer_validation[
                "layer"
            ].eq(
                "province"
            ),
            "feature_n",
        ].iloc[
            0
        ]
    )
    if (
        (
            layer_validation[
                "layer"
            ].eq(
                "province"
            )
            &
            layer_validation[
                "status"
            ].eq(
                "PASS"
            )
        ).any()
    )
    else np.nan
)


amphoe_n = (
    int(
        layer_validation.loc[
            layer_validation[
                "layer"
            ].eq(
                "amphoe"
            ),
            "feature_n",
        ].iloc[
            0
        ]
    )
    if (
        (
            layer_validation[
                "layer"
            ].eq(
                "amphoe"
            )
            &
            layer_validation[
                "status"
            ].eq(
                "PASS"
            )
        ).any()
    )
    else np.nan
)


tambon_n = (
    int(
        layer_validation.loc[
            layer_validation[
                "layer"
            ].eq(
                "tambon"
            ),
            "feature_n",
        ].iloc[
            0
        ]
    )
    if (
        (
            layer_validation[
                "layer"
            ].eq(
                "tambon"
            )
            &
            layer_validation[
                "status"
            ].eq(
                "PASS"
            )
        ).any()
    )
    else np.nan
)


dataset_summary = pd.DataFrame([
    {
        "dataset":
            "PCD annual Excel",

        "records_or_files":
            int(
                excel_validation[
                    "status"
                ]
                .eq(
                    "PASS"
                )
                .sum()
            ),

        "unit":
            "files",
    },

    {
        "dataset":
            "PCD/Air4Thai stations",

        "records_or_files":
            len(
                pcd_stations
            ),

        "unit":
            "stations",
    },

    {
        "dataset":
            "OGIMET/WMO stations",

        "records_or_files":
            len(
                ogimet_stations
            ),

        "unit":
            "stations",
    },

    {
        "dataset":
            "Province",

        "records_or_files":
            province_n,

        "unit":
            "features",
    },

    {
        "dataset":
            "Amphoe",

        "records_or_files":
            amphoe_n,

        "unit":
            "features",
    },

    {
        "dataset":
            "Tambon",

        "records_or_files":
            tambon_n,

        "unit":
            "features",
    },
])


display(
    dataset_summary
)

,dataset,records_or_files,unit
0,PCD annual Excel,5,files
1,PCD/Air4Thai stations,173,stations
2,OGIMET/WMO stations,130,stations
3,Province,77,features
4,Amphoe,928,features
5,Tambon,7425,features


# 1.18 Final Student Readiness Test

Notebook จะให้สถานะ `TEACHING_DATASET_READY` เมื่อ:

1. SHA256 ของ package ทุกไฟล์ตรง
2. Excel 2021–2025 เปิดได้ครบ
3. PCD station table มีข้อมูลและพิกัด
4. OGIMET/WMO station table มีข้อมูลและพิกัด
5. Province = 77
6. GeoPackage expected layers มีครบ
7. CRS ของทุก layer เป็น EPSG:4326
8. invalid geometry = 0

ถ้าไม่ผ่าน จะขึ้น `CHECK_DATASET`
และไม่ควรเริ่ม Notebook 02 จนกว่าจะทราบสาเหตุ

In [25]:
# CELL 25 — Final readiness test

sha_pass = bool(
    download_validation[
        "sha256_ok"
    ].all()
)

excel_pass = (
    len(
        excel_validation
    ) == 5
    and excel_validation[
        "status"
    ].eq(
        "PASS"
    ).all()
)

pcd_pass = (
    len(
        pcd_stations
    ) > 0
    and int(
        (
            ~pcd_coordinate_valid
        ).sum()
    ) == 0
)

ogimet_pass = (
    len(
        ogimet_stations
    ) > 0
    and int(
        (
            ~ogimet_coordinate_valid
        ).sum()
    ) == 0
)

province_pass = (
    province_n == 77
)

layers_pass = (
    len(
        layer_validation
    )
    == len(
        EXPECTED_LAYERS
    )
    and layer_validation[
        "status"
    ].eq(
        "PASS"
    ).all()
)

crs_pass = (
    layer_validation.loc[
        layer_validation[
            "status"
        ].eq(
            "PASS"
        ),
        "crs",
    ]
    .astype(str)
    .str.contains(
        "4326",
        regex=False,
    )
    .all()
)

geometry_pass = (
    pd.to_numeric(
        layer_validation.loc[
            layer_validation[
                "status"
            ].eq(
                "PASS"
            ),
            "invalid_geometry_n",
        ],
        errors="coerce",
    )
    .fillna(
        0
    )
    .eq(
        0
    )
    .all()
)


student_readiness = pd.DataFrame([
    {
        "check":
            "Package SHA256",

        "status":
            (
                "PASS"
                if sha_pass
                else "FAIL"
            ),
    },

    {
        "check":
            "PCD Excel 2021–2025",

        "status":
            (
                "PASS"
                if excel_pass
                else "FAIL"
            ),
    },

    {
        "check":
            "PCD station metadata",

        "status":
            (
                "PASS"
                if pcd_pass
                else "FAIL"
            ),
    },

    {
        "check":
            "OGIMET/WMO station metadata",

        "status":
            (
                "PASS"
                if ogimet_pass
                else "FAIL"
            ),
    },

    {
        "check":
            "Province count = 77",

        "status":
            (
                "PASS"
                if province_pass
                else "FAIL"
            ),
    },

    {
        "check":
            "Expected GeoPackage layers",

        "status":
            (
                "PASS"
                if layers_pass
                else "FAIL"
            ),
    },

    {
        "check":
            "Canonical CRS EPSG:4326",

        "status":
            (
                "PASS"
                if crs_pass
                else "FAIL"
            ),
    },

    {
        "check":
            "Valid geometry",

        "status":
            (
                "PASS"
                if geometry_pass
                else "FAIL"
            ),
    },
])


display(
    student_readiness
)


all_ready = (
    student_readiness[
        "status"
    ]
    .eq(
        "PASS"
    )
    .all()
)


if all_ready:

    FINAL_STATUS = (
        "TEACHING_DATASET_READY"
    )

else:

    FINAL_STATUS = (
        "CHECK_DATASET"
    )


print(
    "\n=================================="
)

print(
    FINAL_STATUS
)

print(
    "=================================="
)

print(
    "\nCanonical data folder:"
)

print(
    DATASET_DIR
)

,check,status
0,Package SHA256,PASS
1,PCD Excel 2021–2025,PASS
2,PCD station metadata,PASS
3,OGIMET/WMO station metadata,PASS
4,Province count = 77,PASS
5,Expected GeoPackage layers,PASS
6,Canonical CRS EPSG:4326,PASS
7,Valid geometry,PASS



TEACHING_DATASET_READY

Canonical data folder:
/content/drive/MyDrive/Teaching_PCD_Environmental_GIS/01_course_data/v1/dataset


# แบบฝึกหัดท้าย Notebook 01

## Exercise 1 — Manifest

เปิด `dataset_packages.csv` แล้วตอบ:

1. มี package กี่ไฟล์?
2. package ใดมีขนาดใหญ่ที่สุด?
3. SHA256 มีความยาวกี่ตัวอักษร?
4. ทำไมเราไม่ควรใช้เพียง filename เพื่อตรวจว่าเป็นไฟล์เดียวกัน?

---

## Exercise 2 — Raw vs Processed

ยกตัวอย่าง:

```text
Raw data 3 ไฟล์
Processed data 3 ไฟล์
Metadata 3 ไฟล์
```

จาก dataset inventory

แล้วอธิบายความแตกต่าง

---

## Exercise 3 — Geometry

จาก GeoPackage ตอบว่า:

```text
PCD station        → geometry ชนิดใด?
OGIMET station     → geometry ชนิดใด?
Province           → geometry ชนิดใด?
Amphoe             → geometry ชนิดใด?
Tambon             → geometry ชนิดใด?
```

---

## Exercise 4 — CRS

ตอบว่า:

1. CRS ของ canonical GIS คืออะไร?
2. EPSG:4326 ใช้หน่วยเป็นเมตรหรือองศา?
3. ทำไมเรายังไม่ควรใช้ `distance()` บน EPSG:4326 โดยตรง?

คำถามที่ 3 จะกลับมาอีกครั้งใน Notebook 03–04

---

## Exercise 5 — Provenance

เลือก dataset 3 ชนิดจาก `data_provenance.csv`

รายงาน:

```text
dataset
primary_source
source_location
processing
canonical_output
```

---

# สิ่งที่ต้องจำ

1. Canonical dataset ทำให้ทุกคนใช้ข้อมูล version เดียวกัน
2. SHA256 ใช้ตรวจว่าไฟล์ที่ได้ตรงกับต้นฉบับ
3. ZIP แตกสำเร็จไม่ได้แปลว่าข้อมูลพร้อมใช้
4. Excel ต้องตรวจ schema
5. station table ต้องตรวจ ID และ coordinate
6. spatial data ต้องตรวจ geometry และ CRS
7. GeoPackage สามารถเก็บหลาย GIS layers ในไฟล์เดียว
8. raw data, processed data และ metadata มีหน้าที่ต่างกัน
9. provenance เป็นส่วนหนึ่งของ reproducible research
10. จากนี้เป็นต้นไปใช้ข้อมูลจาก course repository เท่านั้น

# Notebook ต่อไป

## Notebook 02 — Pandas and Air-Quality Data Foundations

ชื่อ:

```text
02_pandas_airquality_data_foundations.ipynb
```

Notebook 02 จะเริ่มจาก:

```text
2021.xlsx–2025.xlsx
```

โดยยังไม่เน้น GIS

ลำดับการเรียน:

```text
Excel
 ↓
DataFrame
 ↓
Rows / Columns
 ↓
Datetime
 ↓
Station
 ↓
Pollutants
 ↓
Missing / Duplicate
 ↓
Filtering
 ↓
Grouping
 ↓
Daily / Monthly Aggregation
 ↓
Time-Series Visualization
```

ปลายบทจะเกิดคำถามสำคัญ:

> เรารู้แล้วว่าค่ามลพิษเปลี่ยนแปลง “เมื่อไร”  
> แต่สถานีเหล่านี้อยู่ “ที่ไหน”?

คำถามนี้จะนำเข้าสู่ GIS ใน Notebook 03